# Notebook 08b — XGBoost seed robustness (separate from Notebook 08)

**Purpose (supervisor decision 4B).** Retrain the local and centralized XGBoost models with five random seeds to measure how much the reported scores depend on training randomness. Notebook 08 and its outputs are not modified.

**Why a fixed XGBoost version.** The published XGBoost results were produced with XGBoost 3.2.0. A re-run with XGBoost 3.4.1 gave different scores, while all other classical models reproduced exactly. This notebook therefore installs XGBoost 3.2.0.

**Safety rule.** Section 3 must reproduce all six published seed-42 scores. If any score differs, the notebook stops and nothing else is computed.

**Before running:** Settings → Internet **On**. Add Input → the **original Notebook 02 output** (the one containing `preprocess_manifest.json` and `tensors/`).

In [1]:
# 1 — Install the XGBoost version used for the published results
%pip install -q "xgboost==3.2.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 2 — Imports and version check (stops if the wrong version is loaded)
import os, sys, shutil, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
import xgboost, sklearn
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score

print("python      :", sys.version.split()[0])
print("xgboost     :", xgboost.__version__)
print("scikit-learn:", sklearn.__version__)
print("numpy       :", np.__version__)
assert xgboost.__version__ == "3.2.0", "Wrong XGBoost version loaded: restart the session (Run > Restart) and run again from cell 1."

python      : 3.13.16
xgboost     : 3.2.0
scikit-learn: 1.6.1
numpy       : 2.1.3


In [3]:
# 3 — Load the ORIGINAL Notebook 02 tensors (same loading and flattening as Notebook 08)
ACTIVE_CLIENTS = ["England", "Ireland", "USA"]

def find_dir(marker, start="/kaggle/input"):
    for root, dirs, files in os.walk(start):
        if marker in files:
            return Path(root)
    return None

N02_DIR = find_dir("preprocess_manifest.json")
assert N02_DIR is not None, "Notebook 02 output not found under /kaggle/input."
TENSOR_DIR = N02_DIR / "tensors"
print("Tensors from:", TENSOR_DIR)

data = {}
for c in ACTIVE_CLIENTS:
    a = np.load(TENSOR_DIR / f"{c.lower()}_stage1_sequences.npz")
    data[c] = {}
    for s in ["train", "test"]:
        X = a[f"X_{s}"]
        data[c][s] = {"X": X.reshape(X.shape[0], -1), "y": a[f"y_{s}"]}
    print(f"{c:8s} train={data[c]['train']['X'].shape}  test={data[c]['test']['X'].shape}")

X_train_central = np.concatenate([data[c]["train"]["X"] for c in ACTIVE_CLIENTS], axis=0)
y_train_central = np.concatenate([data[c]["train"]["y"] for c in ACTIVE_CLIENTS], axis=0)
print("Pooled train:", X_train_central.shape)

Tensors from: /kaggle/input/datasets/sarahaddouda/preprocess-outputs/tensors
England  train=(76530, 96)  test=(11942, 96)
Ireland  train=(27778, 96)  test=(5887, 96)
USA      train=(30222, 96)  test=(1979, 96)
Pooled train: (134530, 96)


In [4]:
# 4 — Same XGBoost settings and training procedure as Notebook 08
def make_xgb(seed):
    return XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                         subsample=0.85, colsample_bytree=0.85,
                         objective="multi:softprob", eval_metric="mlogloss",
                         tree_method="hist", random_state=seed, n_jobs=-1)

def evaluate(model, client):
    X, y = data[client]["test"]["X"], data[client]["test"]["y"]
    p = np.asarray(model.predict(X)).reshape(-1).astype(int)
    present = sorted(np.unique(y).tolist())
    return {"Macro_F1_present": f1_score(y, p, labels=present, average="macro", zero_division=0),
            "Accuracy": accuracy_score(y, p), "Balanced_accuracy": balanced_accuracy_score(y, p)}

def run_seed(seed):
    rows = []
    for c in ACTIVE_CLIENTS:                                   # local
        X, y = data[c]["train"]["X"], data[c]["train"]["y"]
        m = make_xgb(seed); m.fit(X, y, sample_weight=compute_sample_weight(class_weight="balanced", y=y))
        rows.append({"Strategy": "Local", "Client": c, "Seed": seed, **evaluate(m, c)})
    m = make_xgb(seed)                                         # centralized
    m.fit(X_train_central, y_train_central,
          sample_weight=compute_sample_weight(class_weight="balanced", y=y_train_central))
    for c in ACTIVE_CLIENTS:
        rows.append({"Strategy": "Centralized", "Client": c, "Seed": seed, **evaluate(m, c)})
    return rows

In [5]:
# 5 — Reproduction check: seed 42 must give the six published scores. Otherwise STOP.
REPORTED = {("Local", "England"): 0.7145, ("Local", "Ireland"): 0.4995, ("Local", "USA"): 0.7889,
            ("Centralized", "England"): 0.6699, ("Centralized", "Ireland"): 0.5381, ("Centralized", "USA"): 0.7735}

seed42 = pd.DataFrame(run_seed(42))
seed42["Reported"] = [REPORTED[(s, c)] for s, c in zip(seed42.Strategy, seed42.Client)]
seed42["Match"] = (seed42.Macro_F1_present.round(4) - seed42.Reported).abs() < 1e-9
display(seed42[["Strategy", "Client", "Macro_F1_present", "Reported", "Match"]].round(4))

if not seed42.Match.all():
    raise RuntimeError("STOP: seed 42 does not reproduce the published XGBoost scores. "
                       "Do not run the other seeds; report this table.")
print("Reproduction check passed: all six published seed-42 scores reproduced.")

,Strategy,Client,Macro_F1_present,Reported,Match
0,Local,England,0.7145,0.7145,True
1,Local,Ireland,0.4995,0.4995,True
2,Local,USA,0.7889,0.7889,True
3,Centralized,England,0.6699,0.6699,True
4,Centralized,Ireland,0.5381,0.5381,True
5,Centralized,USA,0.7735,0.7735,True


Reproduction check passed: all six published seed-42 scores reproduced.


In [6]:
# 6 — The four additional seeds (only reached if the check passed)
OTHER_SEEDS = [1, 2, 3, 4]
rows = seed42.drop(columns=["Reported", "Match"]).to_dict("records")
for s in OTHER_SEEDS:
    rows += run_seed(s)
    print("seed", s, "done")
seeds_df = pd.DataFrame(rows)

summary = (seeds_df.groupby(["Strategy", "Client"])["Macro_F1_present"]
           .agg(Mean="mean", SD="std", Min="min", Max="max").reset_index())
summary["Seed42_published"] = [REPORTED[(s, c)] for s, c in zip(summary.Strategy, summary.Client)]
display(summary.round(4))

local_mean = seeds_df[seeds_df.Strategy == "Local"].groupby("Seed")["Macro_F1_present"].mean()
print("Local XGBoost mean across the three networks, per seed:", local_mean.round(4).to_dict())
print("Mean = %.4f | SD = %.4f | min = %.4f | max = %.4f" % (local_mean.mean(), local_mean.std(), local_mean.min(), local_mean.max()))

seed 1 done
seed 2 done
seed 3 done
seed 4 done


,Strategy,Client,Mean,SD,Min,Max,Seed42_published
0,Centralized,England,0.6740,0.0034,0.6699,0.6786,0.6699
1,Centralized,Ireland,0.5319,0.0113,0.5206,0.5472,0.5381
2,Centralized,USA,0.7725,0.0039,0.7683,0.7762,0.7735
3,Local,England,0.7030,0.0088,0.6921,0.7145,0.7145
4,Local,Ireland,0.5003,0.0012,0.4991,0.5022,0.4995
5,Local,USA,0.7896,0.0018,0.7880,0.7927,0.7889


Local XGBoost mean across the three networks, per seed: {1: 0.6641, 2: 0.6669, 3: 0.6629, 4: 0.6599, 42: 0.6677}
Mean = 0.6643 | SD = 0.0031 | min = 0.6599 | max = 0.6677


In [7]:
# 7 — Save results (Notebook 08 outputs are not touched)
OUT = Path("/kaggle/working/xgboost_seed_robustness"); OUT.mkdir(parents=True, exist_ok=True)
seeds_df.round(6).to_csv(OUT / "xgboost_seeds_all_runs.csv", index=False)
summary.round(6).to_csv(OUT / "xgboost_seeds_summary.csv", index=False)
pd.DataFrame({"package": ["python", "xgboost", "scikit-learn", "numpy"],
              "version": [sys.version.split()[0], xgboost.__version__, sklearn.__version__, np.__version__]}
            ).to_csv(OUT / "environment_versions.csv", index=False)
shutil.make_archive("/kaggle/working/N08b_xgboost_seed_robustness", "zip", OUT)
print("Saved:", sorted(p.name for p in OUT.iterdir()), "| ZIP: /kaggle/working/N08b_xgboost_seed_robustness.zip")

Saved: ['environment_versions.csv', 'xgboost_seeds_all_runs.csv', 'xgboost_seeds_summary.csv'] | ZIP: /kaggle/working/N08b_xgboost_seed_robustness.zip
